# 01 — Explore

**Question.** If reported earnings were unmanaged, the cross-firm distribution of
scaled net income should be smooth. There is no economic reason for a sharp break at
exactly zero. If managers near a small loss use discretion to cross zero, the
distribution should show a *notch*: a deficit of firms just below zero and a surplus
just above.

This notebook does three things, in the order the spec prescribes:

1. **Phase 0** — the crude look. A 300-ticker `yfinance` prototype sample, plotted
   before any statistics are run. The point is to see whether the effect is visible
   at all, and to be honest if it is not.
2. **Phase 2** — the headline sample. SEC Financial Statement Data Sets, with the
   full exclusion log printed.
3. Sanity checks on the constructed variables (the lagged denominator especially).

Both panels are read from cached parquet. Nothing here downloads anything —
`python src/build_panel.py` produces the caches once.

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(20250811)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

DATA = ROOT / "data"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("project root:", ROOT)

project root: /Users/riddhi/Downloads/All Of My Claude Skills/FinanceDSProject


## Phase 0 — the crude look (yfinance prototype)

Sample: 300 tickers drawn at random (seed 42) from EDGAR's ticker universe. Random
sampling matters — an index like the S&P 500 contains almost no observations near
zero earnings, which is the entire region of interest.

This sample is thin and survivorship-biased (yfinance serves currently-listed
tickers only). It is a sanity check, not the result.

In [2]:
from src.panel import restrict_window
from src.discontinuity import run_all_widths, summarize
from src.plotting import headline_histogram

yf_panel = pd.read_parquet(DATA / "panel_yfinance.parquet")
yf_log = pd.read_csv(DATA / "drop_log_yfinance.csv")

print(yf_log[["step", "rows_before", "rows_dropped", "rows_after", "firms_after"]].to_string(index=False))

                                               step  rows_before  rows_dropped  rows_after  firms_after
                                  0. raw firm-years         1416             0        1416          300
                                2. drop missing SIC         1416           362        1054          222
            3. drop financial firms (SIC 6000-6999)         1054           255         799          167
                  4. drop utilities (SIC 4900-4949)          799            40         759          158
                         5. drop missing net income          759           139         620          158
6. drop missing or non-positive lagged total assets          620           163         457          156
        7. size floor: lagged assets >= $10,000,000          457            56         401          140
                 8. one filing per firm-fiscal-year          401             0         401          140


In [3]:
yf_window, yf_info = restrict_window(yf_panel, "roa")
print(yf_info)
print()
print(summarize(run_all_widths(yf_window["roa"])))

{'column': 'roa', 'with_measure': 401, 'inside_window': 204, 'outside_window': 197, 'missing_measure': 0}

width=0.0025  N=204  below zero: 1 vs 3.0 expected (deficit, z=-1.28, p=0.201)  |  above zero: 3 vs 3.0 expected (deficit, z=+0.00, p=1)
width=0.0050  N=204  below zero: 4 vs 6.5 expected (deficit, z=-0.95, p=0.343)  |  above zero: 8 vs 2.0 expected (surplus, z=+2.04, p=0.0415)
width=0.0100  N=204  below zero: 9 vs 8.5 expected (surplus, z=+0.14, p=0.888)  |  above zero: 8 vs 12.0 expected (deficit, z=-1.11, p=0.267)


In [4]:
fig, _ = headline_histogram(
    yf_window["roa"], 0.005,
    source="yfinance (300-ticker prototype sample)",
    n_firms=yf_window["firm_id"].nunique(),
    period=f"fiscal years {int(yf_window.fiscal_year.min())}-{int(yf_window.fiscal_year.max())}",
    title="Phase 0 prototype: scaled net income, 300-ticker yfinance sample",
    out=FIGS / "phase0_yfinance_histogram.png")
fig

<Figure size 1012x616 with 1 Axes>

**Phase 0 verdict.** Read the numbers above before reading anything into the
picture. With only a couple of hundred observations inside the window, individual
bins hold single-digit counts and the neighbour-predicted expectation is itself
noisy — no configuration of this sample could produce a credible test. The bar just
below zero sits below its neighbours at the two finer widths, but nothing here is
distinguishable from sampling variation.

The correct conclusion is *"too small to tell"*, which is exactly why the spec treats
this as a gate and not a result. On to the SEC sample.

## Phase 2 — the headline sample (SEC Financial Statement Data Sets)

Source: quarterly flat files derived from XBRL filings, at
`https://www.sec.gov/files/dera/data/financial-statement-data-sets/{YYYY}q{Q}.zip`.
Schema verified against the live 2026q1 archive.

Only **10-K** submissions are used. A 10-K carries the comparative balance sheet, so a
single filing supplies both current-year and prior-year total assets — which is where
the lagged denominator comes from, with no cross-filing join required.

In [5]:
panel = pd.read_parquet(DATA / "panel.parquet")
drop_log = pd.read_csv(DATA / "drop_log.csv")

print(f"{len(panel):,} firm-years | {panel['firm_id'].nunique():,} firms | "
      f"fiscal years {int(panel.fiscal_year.min())}-{int(panel.fiscal_year.max())}")
print()
print(drop_log[["step", "rows_before", "rows_dropped", "rows_after", "firms_after"]].to_string(index=False))

31,240 firm-years | 5,366 firms | fiscal years 2016-2025

                                               step  rows_before  rows_dropped  rows_after  firms_after
                                  0. raw firm-years        57407             0       57407        10304
              1. restrict to fiscal years 2016-2025        57407           141       57266        10276
                                2. drop missing SIC        57266           550       56716        10078
            3. drop financial firms (SIC 6000-6999)        56716         14590       42126         7251
                  4. drop utilities (SIC 4900-4949)        42126          1404       40722         7034
                         5. drop missing net income        40722           703       40019         6979
6. drop missing or non-positive lagged total assets        40019           764       39255         6882
        7. size floor: lagged assets >= $10,000,000        39255          8011       31244         5366
      

### Every exclusion, with its stated rationale

In [6]:
for _, r in drop_log.iterrows():
    print(f"{r['step']}\n    dropped {r['rows_dropped']:,} -> {r['rows_after']:,} remain")
    print(f"    rationale: {r['rationale']}\n")

0. raw firm-years
    dropped 0 -> 57,407 remain
    rationale: All 10-K firm-years assembled from the source.

1. restrict to fiscal years 2016-2025
    dropped 141 -> 57,266 remain
    rationale: Only these fiscal years are fully covered by the downloaded filing quarters. Stragglers outside the range are delinquent filers whose years are represented by a handful of firms, not a usable cross-section.

2. drop missing SIC
    dropped 550 -> 56,716 remain
    rationale: Industry filters below cannot be applied without an industry code.

3. drop financial firms (SIC 6000-6999)
    dropped 14,590 -> 42,126 remain
    rationale: Bank and insurer balance sheets are structurally different; total assets does not mean the same thing, so scaling by it is not comparable.

4. drop utilities (SIC 4900-4949)
    dropped 1,404 -> 40,722 remain
    rationale: Rate-regulated returns make utility earnings mechanically smooth near a target; conventional in this literature. Optional filter, applied here.

### Sanity check: the denominator really is lagged

`ROA_it = NetIncome_it / TotalAssets_i,t-1`. Using end-of-year assets would put the
current year's earnings into the denominator — the single most common way to ruin
this measure. The check below confirms the two differ, and shows how much the choice
would move the answer.

In [7]:
both = panel[panel["assets"].notna() & panel["assets_lag"].notna()].copy()
both["roa_lagged"] = both["net_income"] / both["assets_lag"]
both["roa_end_of_year_WRONG"] = both["net_income"] / both["assets"]

print(f"lagged == end-of-year in only {np.isclose(both.assets, both.assets_lag).mean():.2%} of firm-years")
print(f"median |difference| in the resulting ratio: "
      f"{(both.roa_lagged - both.roa_end_of_year_WRONG).abs().median():.4f}")
print()
print(both[["assets_lag", "assets", "roa_lagged", "roa_end_of_year_WRONG"]].describe().round(3).to_string())

lagged == end-of-year in only 0.02% of firm-years
median |difference| in the resulting ratio: 0.0093

         assets_lag        assets  roa_lagged  roa_end_of_year_WRONG
count  3.123500e+04  3.123500e+04   31235.000              31235.000
mean   6.125806e+09  6.474707e+09      -0.150                    NaN
std    2.479985e+10  2.622136e+10       0.701                    NaN
min    1.000371e+07  0.000000e+00     -49.781                   -inf
25%    1.098095e+08  1.218275e+08      -0.215                 -0.204
50%    6.150560e+08  6.822780e+08       0.000                  0.000
75%    2.925910e+09  3.139473e+09       0.065                  0.060
max    6.248940e+11  8.180420e+11      15.078                    inf


### Sanity check: coverage of each measure

The robustness checks need alternate denominators. Market value of equity is **not**
available in the SEC flat files (they carry no price data), so book equity stands in
for it on this sample and market value is tested on the yfinance sample instead. That
substitution is a documented deviation from the spec, not a silent one.

In [8]:
cov = pd.DataFrame({
    "non-null": panel[["net_income", "assets_lag", "revenue", "cfo", "equity_lag", "market_equity"]].notna().sum(),
})
cov["share of panel"] = (cov["non-null"] / len(panel)).round(3)
print(cov.to_string())
print()
print(panel[["roa", "ni_revenue", "ni_equity_lag", "cfo_at"]].describe().round(3).to_string())

               non-null  share of panel
net_income        31240           1.000
assets_lag        31240           1.000
revenue           28332           0.907
cfo               31199           0.999
equity_lag        30589           0.979
market_equity         0           0.000

             roa  ni_revenue  ni_equity_lag     cfo_at
count  31240.000   28044.000      27337.000  31199.000
mean      -0.150     -16.367         -1.346     -0.048
std        0.701     358.188        147.208      0.373
min      -49.781  -31028.750     -24164.454    -11.531
25%       -0.215      -0.233         -0.384     -0.098
50%        0.000       0.011          0.011      0.053
75%        0.065       0.081          0.156      0.118
max       15.078    8437.860       1548.205     13.443


### Composition of the sample

In [9]:
print("firm-years by fiscal year:")
print(panel["fiscal_year"].value_counts().sort_index().to_string())
print()
print("top industry divisions (2-digit SIC):")
print((panel["sic"] // 100).value_counts().head(10).to_string())
print()
print("years per firm (pooling is acknowledged, not hidden):")
print(panel.groupby("firm_id").size().describe().round(2).to_string())

firm-years by fiscal year:
fiscal_year
2016    2626
2017    3027
2018    3063
2019    3037
2020    3108
2021    3490
2022    3609
2023    3391
2024    3193
2025    2696

top industry divisions (2-digit SIC):
sic
28.0    6530
73.0    4553
38.0    2343
36.0    2200
35.0    1429
13.0    1342
37.0     948
48.0     772
80.0     721
20.0     672

years per firm (pooling is acknowledged, not hidden):
count    5366.00
mean        5.82
std         3.26
min         1.00
25%         3.00
50%         6.00
75%         9.00
max        10.00


**Note on pooling.** Multiple years per firm are kept. This inflates N relative to
the number of independent units and the tests below therefore overstate precision
somewhat. The alternative — one year per firm — would cut the sample by roughly the
average number of years per firm for no gain in the question being asked. Pooling is
the simplest defensible choice and is stated rather than hidden.

### The distribution, before any test

Fine bins, zero marked, no statistics yet. Look at it first.

In [10]:
window, info = restrict_window(panel, "roa")
print(info)

fig, table = headline_histogram(
    window["roa"], 0.0025,
    source="SEC Financial Statement Data Sets (10-K filings)",
    n_firms=window["firm_id"].nunique(),
    period=f"fiscal years {int(window.fiscal_year.min())}-{int(window.fiscal_year.max())}",
    title="First look: scaled net income near zero, SEC sample (fine bins)",
    out=FIGS / "explore_sec_fine_bins.png")
fig

{'column': 'roa', 'with_measure': 31240, 'inside_window': 16018, 'outside_window': 15222, 'missing_measure': 0}


<Figure size 1012x616 with 1 Axes>

In [11]:
print(table[["left", "right", "count", "expected", "z"]]
      .iloc[len(table)//2 - 6: len(table)//2 + 6].to_string(index=False))

   left   right  count  expected         z
-0.0150 -0.0125    158     195.5 -2.363190
-0.0125 -0.0100    200     178.5  1.274078
-0.0100 -0.0075    199     205.0 -0.348505
-0.0075 -0.0050    210     206.0  0.228070
-0.0050 -0.0025    213     228.0 -0.837320
-0.0025  0.0000    246     252.0 -0.314374
 0.0000  0.0025    291     252.5  1.906074
 0.0025  0.0050    259     285.0 -1.312794
 0.0050  0.0075    279     256.5  1.127422
 0.0075  0.0100    254     282.0 -1.425148
 0.0100  0.0125    285     279.5  0.270049
 0.0125  0.0150    305     285.5  0.932968
